# Lab 4: Lexical Text Features

**Business question:** Which words and phrases give a business team useful evidence that people can check?

This lab has three parts:

| Part | Data | Time | What you hand in |
|---|---|---|---|
| 1 Guided core | 24 customer messages | about 35 min | Run outputs, saved files and the checkpoint |
| 2 Practice | Women's clothing reviews | about 30–45 min | Three exercises and the interpretation |
| 3 Challenge | Both | optional | Nothing required |

Graded with the course Lab Notebook rubric in Canvas.

## Part 1 Guided core

### 1.1 Load the tools

Select the `ban6403` kernel in VS Code, then run the cell below. It loads pandas, numpy and the two scikit-learn tools that turn text into features. No output means it worked. If an import fails, check that `ban6403` is selected and the packages are installed in it. The Module 0 readiness notebook shows what to fix.

In [ ]:
# Path builds file paths; numpy and pandas hold numbers and tables.
from pathlib import Path
import numpy as np
import pandas as pd
# The two scikit-learn tools that turn text into count and TF-IDF features.
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

### 1.2 Point to the data folder

The notebook runs from the `notebooks` folder, so the data folder is one level up: `../data`. This cell shows no output.

In [ ]:
# ".." means one folder up: the data folder next to notebooks/.
DATA = Path("../data")

### 1.3 Read the customer messages

We read the 24 feedback messages and the partition file from Module 2. The original files are never changed.

In [ ]:
feedback = pd.read_csv(DATA / "raw/customer_feedback_m01.csv")
partition = pd.read_csv(DATA / "partitions/m02_partition.csv")
# How many messages are in each partition.
partition["partition"].value_counts()

### 1.4 Keep only development messages

We build features from the 13 **development** messages only. The protected messages are saved for later evaluation, so we do not look at them while choosing settings.

In [ ]:
# IDs of the development messages.
dev_ids = partition.loc[partition["partition"].eq("development"), "feedback_id"]

# Use the IDs as row labels and keep the text of those 13 messages.
messages = feedback.set_index("feedback_id")
dev = messages.loc[dev_ids, ["text"]].copy()
dev.shape

### 1.5 Recreate the light text view from Module 3

This normalizes Unicode and collapses extra spaces. The original `text` column stays next to the new `light_text` column.

In [ ]:
# Same light view as Module 3: NFC, then split on whitespace and join with single spaces.
dev["light_text"] = dev["text"].str.normalize("NFC").str.split().str.join(" ")
dev.loc[["FB003", "FB005", "FB006"], ["text", "light_text"]]

### 1.6 Count features

`CountVectorizer` learns one column per word and counts how often each word appears in each message. We keep common words (`stop_words=None`) because words like *no* and *not* change the meaning of a service message. The default token rule still applies: it splits at punctuation and drops one-character tokens, so *don't* becomes *don* (Challenge A shows this).

scikit-learn tools work in two steps. `fit` learns the vocabulary from the text. `transform` turns text into a matrix with one row per message and one column per word. `fit_transform` does both at once. The result is a **sparse matrix**: it stores only the cells that are not zero.

In [ ]:
# Lowercase text and keep stop words; the default token rule still applies.
counts = CountVectorizer(lowercase=True, stop_words=None)

# Learn the vocabulary (fit) and count each word in each message (transform).
X_count = counts.fit_transform(dev["light_text"])

# (rows, columns) = (messages, words in the vocabulary).
X_count.shape

### 1.7 Look at the vocabulary

You should see 69 word columns. Feature names are listed alphabetically; the order says nothing about importance.

In [ ]:
# The word that belongs to each column, in column order.
count_features = counts.get_feature_names_out()

# [:12] shows the first 12 words.
count_features[:12]

### 1.8 Make a readable table

This puts message IDs on the rows and words on the columns, without expanding the whole matrix into memory.

In [ ]:
# Wrap the sparse matrix in a DataFrame: message IDs as rows, words as columns.
count_view = pd.DataFrame.sparse.from_spmatrix(
    X_count, index=dev.index, columns=count_features
).fillna(0)  # empty cells would show as NaN; fillna(0) shows them as 0

### 1.9 Read a few counts

Compare these counts with the original messages FB003, FB005 and FB006. A zero only means the word is absent.

In [ ]:
# Three messages and four words from the count table.
count_view.loc[["FB003", "FB005", "FB006"], ["refund", "no", "not", "start"]]

### 1.10 How sparse is the matrix?

`nnz` is the number of nonzero cells. Expect 96 nonzero cells out of 13 × 69, so about 89% of the matrix is zeros. Real text data is even sparser, which is why scikit-learn stores it in a sparse format.

In [ ]:
# Share of zero cells = 1 - nonzero cells / all cells.
all_cells = X_count.shape[0] * X_count.shape[1]
zero_share = 1 - X_count.nnz / all_cells
print("nonzero entries:", X_count.nnz)
print("zero share:", round(zero_share, 3))

### 1.11 TF-IDF weights

`TfidfVectorizer` starts from counts and lowers the weight of words that appear in many messages. Each row is then scaled to length 1. A TF-IDF value is a coordinate in this table. It is not a probability, and it does not measure business importance.

In [ ]:
# ngram_range=(1, 1): single words only. Same fit_transform step as for counts.
unigram = TfidfVectorizer(ngram_range=(1, 1), stop_words=None)
X_uni = unigram.fit_transform(dev["light_text"])
X_uni.shape

### 1.12 Make a readable TF-IDF table

Same as 1.8, now with the TF-IDF weights instead of counts. This cell shows no output.

In [ ]:
# Same table as in 1.8, now with TF-IDF weights.
uni_features = unigram.get_feature_names_out()
uni_view = pd.DataFrame.sparse.from_spmatrix(
    X_uni, index=dev.index, columns=uni_features
).fillna(0)

### 1.13 Compare weights with counts

Look at the same messages and words as in 1.9. In FB005, *refund* and *no* get the same weight (about 0.354) because each appears in three development messages. FB003 is not longer (7 words against 8). Its *refund* weight is a little lower (about 0.325) because FB003 has four words that no other development message uses (*arrange*, *headset*, *stopped*, *working*). Their high IDF makes the row longer before scaling, so *refund* gets a smaller share. A TF-IDF weight depends on the word's count, its IDF and the other words in the same row.

In [ ]:
# The same messages and words as in 1.9, now as TF-IDF weights.
uni_view.loc[["FB003", "FB005", "FB006"], ["refund", "no", "not", "start"]]

### 1.14 Add two-word phrases

Change only one setting: `ngram_range=(1, 2)` keeps single words and adds adjacent two-word phrases. Expect 147 columns instead of 69.

In [ ]:
# ngram_range=(1, 2): single words and two-word phrases.
bigram = TfidfVectorizer(ngram_range=(1, 2), stop_words=None)
X_bi = bigram.fit_transform(dev["light_text"])
X_bi.shape

### 1.15 Make a readable phrase table

Same as 1.12, for the words and phrases. This cell shows no output.

In [ ]:
# Same table again, now with words and two-word phrases as columns.
bi_features = bigram.get_feature_names_out()
bi_view = pd.DataFrame.sparse.from_spmatrix(
    X_bi, index=dev.index, columns=bi_features
).fillna(0)

### 1.16 Phrases that carry meaning

*not start* (FB006) and *no action* (FB005 and FB020) keep the negation next to the word it changes. Single-word features lose that link.

In [ ]:
# The two phrase weights for three messages.
bi_view.loc[["FB005", "FB006", "FB020"], ["no action", "not start"]]

### 1.17 New text uses the frozen vocabulary

After fitting, the vocabulary is frozen: it stays fixed and never learns new words. `transform` applies the same columns to new text. The word *hyperflux* never appeared in the development messages, so `transform` ignores it and no new column appears. This is why we fit on development data once and then only transform.

In [ ]:
# A new message: transform only, no refitting.
probe_text = ["Hyperflux charger failed."]
X_probe = unigram.transform(probe_text)

# vocabulary_ is a dictionary from each known word to its column number.
print("hyperflux in vocabulary:", "hyperflux" in unigram.vocabulary_)
print("probe matrix shape:", X_probe.shape)

### 1.18 Collect a small evidence table

Put the original text next to a few counts and weights so the numbers can be checked against the messages.

In [ ]:
# Start from the two text columns, then add one column per feature.
comparison = dev[["text", "light_text"]].copy()
comparison["refund_count"] = count_view["refund"]
comparison["refund_tfidf"] = uni_view["refund"]

In [ ]:
# Two phrase weights from the bigram table.
comparison["no_action_tfidf"] = bi_view["no action"]
comparison["not_start_tfidf"] = bi_view["not start"]
comparison.loc[["FB005", "FB006", "FB020"]]

### 1.19 Record the two settings

A small table lists both TF-IDF settings and their matrix sizes, so another analyst can repeat them.

In [ ]:
# One row per setting. shape[0] is the number of rows, shape[1] the number of columns.
settings = pd.DataFrame([
    {"setting": "unigram", "ngram_range": "1,1", "rows": X_uni.shape[0], "features": X_uni.shape[1]},
    {"setting": "unigram_bigram", "ngram_range": "1,2", "rows": X_bi.shape[0], "features": X_bi.shape[1]}
])
# The other settings are the same for both rows (the scikit-learn defaults).
settings["policy"] = "lowercase=True; stop_words=None; min_df=1; smooth_idf=True; norm=l2"
settings

### 1.20 Save your outputs

This writes two small CSV files to `outputs/m04`. The raw data is not changed. After saving the notebook, review and commit the changes in GitHub Desktop.

In [ ]:
OUTPUT_DIR = Path("../outputs/m04")
# parents=True also creates outputs/ if needed; exist_ok=True accepts an existing folder.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# The row labels are the message IDs; index_label names that column in the file.
comparison.to_csv(OUTPUT_DIR / "feature_comparison.csv", index_label="feedback_id")
# index=False leaves out pandas' row numbers.
settings.to_csv(OUTPUT_DIR / "representation_settings.csv", index=False)

### Checkpoint

Answer in the cell below (2–3 sentences in total).

1. What information do the phrases *no action* and *not start* add beyond the single words?
2. Name one limitation of phrase features for the service follow-up decision.

**Your answer:**

## Part 2 Practice: real clothing reviews

Now apply the same method to real customer reviews. The merchandising team of an online women's clothing store wants to know two things. Which words set each department's reviews apart? And which words mark unhappy customers?

**The data.** `data/course/clothing_reviews.csv` holds 22,634 real reviews from the public *Women's E-Commerce Clothing Reviews* dataset. The reviews are anonymized: there are no customer names, and the store's name was replaced with the word "retailer". The Kaggle page lists it as free to use (CC0 1.0, a public domain dedication). Each review has a star rating from 1 to 5, a yes/no recommendation and the department of the item. The reviews come with a fixed split into `train`, `valid` and `test`. The data card `data/course/clothing_reviews_README.md` describes every column.

Two things to know about this data:

- Only customers who chose to write a review are here, and most of them were happy: about 77% gave 4 or 5 stars. Counts describe this dataset, not all customers.
- Reviews are short (about 60 words on average) and informal. Expect typos, body measurements and shorthand such as "tts" (true to size).

### 2.1 Load the review file

In [ ]:
# Read the reviews; shape gives (rows, columns).
reviews = pd.read_csv(DATA / "course/clothing_reviews.csv")
reviews.shape

### 2.2 Check the splits and departments

`dropna=False` also counts reviews without a department. There are 13 of them.

In [ ]:
# Count the reviews in each split.
reviews["split"].value_counts()

In [ ]:
# Count the reviews in each department; dropna=False also counts missing departments.
reviews["department"].value_counts(dropna=False)

### 2.3 Separate training and validation rows

We fit features on `train` only. `valid` is used later to show that the frozen vocabulary works on new text. The `test` split stays untouched.

In [ ]:
# Keep the rows of each split; the test rows are not used in this lab.
train = reviews[reviews["split"].eq("train")]
valid = reviews[reviews["split"].eq("valid")]
train.shape, valid.shape

### 2.4 Read a few reviews

Reviews are short, so read these five in full. Notice the sizes and heights, the typos in the last review, and that the text is in lowercase after the first letter. The second review stops in the middle of a word: the source file cut long reviews at about 500 characters.

In [ ]:
# iterrows() gives one row at a time; _ is the row label, which we do not need.
for _, row in train.head(5).iterrows():
    print(row["department"], "|", row["rating"], "stars |", row["review_text"])
    print()

### 2.5 Fit TF-IDF on the training reviews

Same idea as 1.14, with one change for real data: `min_df=5` drops terms that appear in fewer than five reviews. We still keep common words (`stop_words=None`), because *not* changes the meaning of phrases such as *not flattering*.

These are the **Lab 4 Practice settings**: words and two-word phrases (`ngram_range=(1, 2)`), `min_df=5` and no stop words. Lab 5 uses them again. Expect 13,581 rows and 23,828 columns.

In [ ]:
# Words and two-word phrases; min_df=5 keeps terms that appear in at least 5 reviews.
practice_tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=5, stop_words=None)

# fit learns the vocabulary from train only; transform builds the matrix.
X_train = practice_tfidf.fit_transform(train["review_text"])
X_train.shape

### 2.6 Which terms set one department apart?

First we average each column over all training reviews. Then we average it over one department's reviews. Positive differences identify terms with higher average TF-IDF weights in this department than across all training reviews. This is a descriptive comparison, not a significance test. It compares weights, not word counts or the share of reviews that use a term, and those measures can rank terms differently. (If you only look at the department's average, common words like *the* and *it* fill the top of the list.) The subtraction pushes most common words down, but a few, such as *the* and *it*, remain.

Start with Dresses. The top of the list names the item (*dress*, *this dress*). Read further down to see what customers write about: fit (*petite*, *flattering*), what goes underneath (*slip*) and occasions (*wedding*).

In [ ]:
# The term that belongs to each column of X_train.
practice_features = practice_tfidf.get_feature_names_out()

department_name = "Dresses"
# True/False for each training review. to_numpy() makes a plain array,
# which can select rows of the sparse matrix.
in_department = train["department"].eq(department_name).to_numpy()

In [ ]:
# mean(axis=0) averages each column over the rows: one average weight per term.
# np.asarray(...).ravel() turns the result into a flat list of numbers.
overall_mean = np.asarray(X_train.mean(axis=0)).ravel()

# X_train[in_department] keeps only the rows of this department.
department_mean = np.asarray(X_train[in_department].mean(axis=0)).ravel()

# Positive difference: a higher average TF-IDF weight in this department than in all training reviews.
difference = pd.Series(department_mean - overall_mean, index=practice_features)
difference.sort_values(ascending=False).head(25)

### Exercise 1: a second department

Copy the two cells above into the cell below and change `department_name` to another department, exactly as it appears in 2.2. Show its top 25 terms. Which terms only name the item or a brand, and which describe fit, fabric or use?

In [ ]:
# Exercise 1: your code here
# TODO: copy the two cells from 2.6 into this cell.
# TODO: change department_name to another department from 2.2, for example "Bottoms".
# TODO: show the top 25 terms with .head(25).

### Exercise 2: what does a stricter frequency filter remove?

1. Fit a second `TfidfVectorizer` on `train["review_text"]` with the same settings but `min_df=50`. Name it `strict_tfidf`.
2. Show the shape of its matrix and compare the number of columns with 2.5.
3. The cell after yours checks whether four phrases survive with `min_df=5`. After your code works, remove the `#` signs from its two marked lines and run it again. It then adds a column for `min_df=50`.

In [ ]:
# Exercise 2: your code here
# TODO: create strict_tfidf = TfidfVectorizer(...) with the settings of 2.5, but min_df=50.
# TODO: fit it on train["review_text"] with fit_transform and store the matrix as X_strict.
# TODO: show X_strict.shape and compare the number of columns with 2.5.

In [ ]:
phrase_check = pd.DataFrame({"phrase": ["not flattering", "runs small", "wide leg", "pilling"]})
# isin() is True when the phrase is one of the vocabulary terms.
phrase_check["kept_min_df_5"] = phrase_check["phrase"].isin(practice_features)

# TODO: after Exercise 2, remove the # from the next two lines to check the stricter filter.
# strict_features = strict_tfidf.get_feature_names_out()
# phrase_check["kept_min_df_50"] = phrase_check["phrase"].isin(strict_features)
phrase_check

### Exercise 3: apply the frozen vocabulary to validation reviews

Use `practice_tfidf.transform` (not `fit_transform`) on `valid["review_text"]` and show the shape. The number of columns should match `X_train`.

Then run the cell after yours. It lists single words in validation reviews that are **not** columns of the fitted vocabulary. Some never appeared in the training reviews; others appeared in fewer than five. `transform` simply ignores them.

In [ ]:
# Exercise 3: your code here
# TODO: use practice_tfidf.transform (not fit_transform) on valid["review_text"]; name the result X_valid.
# TODO: show X_valid.shape and check that the number of columns matches X_train.

In [ ]:
# The analyzer splits a text into the same words and phrases the vectorizer uses.
analyzer = practice_tfidf.build_analyzer()
# vocabulary_ is a dictionary from each known term to its column number.
vocabulary = practice_tfidf.vocabulary_

def unknown_words(text):
    """Take one review and return a sorted list of its single words that are not in the vocabulary."""
    missing = set()
    for term in analyzer(text):
        # Phrases contain a space; we only look at single words.
        if " " not in term and term not in vocabulary:
            missing.add(term)
    return sorted(missing)

# map() calls unknown_words once per validation review.
unknown = valid["review_text"].map(unknown_words)
# True for each review whose list of unknown words is not empty.
has_unknown = unknown.str.len() > 0
print("validation reviews with at least one unknown word:", has_unknown.sum(), "of", len(valid))
unknown[has_unknown].head(5)

### Practice interpretation

Write 3–5 sentences:

- What do the top terms for your two departments suggest about what customers write about? Which terms only name the item or a brand?
- Which `min_df` setting would you use as a baseline for the merchandising team's word lists, and why? Use the matrix sizes and the phrase check.
- Why must the vocabulary be fit on `train` only? Use the unknown words from Exercise 3 in your answer.

**Your answer:**

### Submit

1. Restart the kernel and run all required cells through Submit. Check that there are no errors. Challenge cells are optional; errors only in them do not reduce your grade.
2. Save the notebook, so the outputs are saved in the file. The saved outputs are what is graded.
3. In GitHub Desktop, write a short summary such as "Lab 4 complete", click **Commit to main**, then **Push origin**.
4. Push before the deadline. You upload nothing to Canvas: the instructor grades the saved notebook in your repository as it is at the deadline.

Record any AI tool use in one line: the tool, what you used it for, and how you checked the result. If you used none, write "None".

**AI use:**

## Part 3 Challenge (optional, not graded)

### A. See the real token boundaries

The default tokenizer splits on punctuation and drops one-character tokens. Look at what happens to a contraction, a code and an amount.

In [ ]:
# The analyzer of the unigram vectorizer from 1.11, applied to one example string.
unigram_analyzer = unigram.build_analyzer()
unigram_analyzer("don't refund AB-12 for $49.99")

### B. Reproduce the IDF numbers from the slides

This uses the three short examples from the video. `idf_` contains the learned IDF values. Row normalization affects TF-IDF rows, not these values, so we only fit the vectorizer and read `idf_`.

In [ ]:
lx_text = ["refund requested", "refund completed", "refund requested"]
# fit learns the vocabulary and the IDF weights; we do not need the matrix here.
lx_vectorizer = TfidfVectorizer()
lx_vectorizer.fit(lx_text)
# idf_ holds one IDF weight per word, in column order.
pd.Series(lx_vectorizer.idf_, index=lx_vectorizer.get_feature_names_out())

### C. Terms that mark low ratings

The merchandising team's second question: which words mark unhappy customers? `chi2` scores how strongly each feature is linked to one of two groups. Here the groups are low-rated reviews (1–2 stars) and high-rated reviews (4–5 stars); 3-star reviews are left out.

For each term, chi-square compares two totals in each group. One is the observed total. The other is the total we would expect if the term were spread over the groups by their size. The larger the gap, the higher the score. The score measures the strength of the link, not its direction: a high score can point to either group.

In [ ]:
from sklearn.feature_selection import chi2

# True/False arrays: 1-2 stars, 4-5 stars, and either of the two (| means "or").
low = train["rating"].le(2).to_numpy()
high = train["rating"].ge(4).to_numpy()
both = low | high

# chi2 compares the two groups for each column. It returns scores and p-values;
# we keep the scores and ignore the p-values (_).
chi_scores, _ = chi2(X_train[both], low[both])
top_terms = pd.Series(chi_scores, index=practice_features).sort_values(ascending=False).head(15)
top_terms

Compare the percentage of low- and high-rated reviews containing each term. A high chi-square score alone does not identify the group. The next cell prints both percentages for the 15 terms above, and for the single word *flattering*.

*not flattering* is in the list. The single word *flattering* also appears in low-rated reviews, partly because some of them say *not flattering*. A single-word feature cannot tell these apart.

In [ ]:
# The 15 terms from the chi-square list, plus the single word "flattering".
terms_to_check = list(top_terms.index) + ["flattering"]
for term in terms_to_check:
    # The column number of this term in X_train.
    column = practice_tfidf.vocabulary_[term]
    # True for each review where the term has a weight above zero.
    # .toarray() turns the sparse column into a normal array; .ravel() makes it flat.
    has_term = (X_train[:, column] > 0).toarray().ravel()
    low_share = has_term[low].mean()
    high_share = has_term[high].mean()
    # :.1% prints a share as a percentage with one decimal.
    print(f"{term}: in {low_share:.1%} of low-rated and {high_share:.1%} of high-rated reviews")

## Further reading

- [scikit-learn: text feature extraction](https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction)
- [scikit-learn: common pitfalls and data leakage](https://scikit-learn.org/stable/common_pitfalls.html)

Next, Module 5 compares these lexical features with sentence embeddings on the same reviews.